## Heuristic Methods
Heuristic methods can be used to solve the general constrained nonlinear optimization problem described above without computing the gradient of the objective function, i.e., they work without derivative information. 


> __Why is that interesting?__
>
> * __Derivatives are a hassle!__ Computing derivatives can be computationally expensive, especially for complex functions or large datasets. There may also be cases where derivatives do not exist or are difficult to compute. For example, we used a smooth barrier function to avoid issues with the derivatives of the augmented objective function. If we had instead used something like $\max(0,g_i(x))^2$ for the inequality constraints, then the derivatives would not be well-defined at the boundary of the feasible region.
> * __Super, let's always use heuristics!__ While heuristic methods can be very effective for solving complex optimization problems, they may not always guarantee convergence to the global optimum or even a local optimum. Further, they tend to be less efficient than gradient-based methods for problems where derivatives are available. We use them when needed, but not always.

There are many heuristic methods, but let's focus on a typical one: simulated annealing. 
___

## Simulated Annealing
Simulated annealing (SA) is inspired by the annealing process in metallurgy, where controlled cooling of a material allows it to reach a low-energy state. SA explores the solution space by accepting both improving and non-improving moves, allowing it to escape local optima. Developed by Kirkpatrick, Gelatt, and Vecchi in 1983, SA is a probabilistic technique that uses a temperature parameter to control the exploration of the solution space.

Let's take a look at the basic steps of the simulated annealing algorithm for minimizing the (augmented) objective function $P_{\mu,\rho}(x)$:
$$
\begin{align*}
    \min_{x\in\mathbb{R}^n}\;P_{\mu,\rho}(x)\;&=f(x)\;-\;\mu\sum_{i=1}^m\ln\bigl(-\,g_i(x)\bigr)\;+\;\frac{1}{2\rho}\sum_{j=1}^p 
    \bigl[h_j(x)\bigr]^2,\quad\text{where}\quad\mu>0,\;\rho>0\\
\end{align*}
$$
where $f(x)$ is the objective function, $g_i(x)$ are the inequality constraints, and $h_j(x)$ are the equality constraints.


__Initialize__: Given an initial solution guess $x_0$, penalty parameters $\mu > 0$ and $\rho > 0$, an initial temperature $T\gets{T_\circ}$, a cooling rate parameter $\alpha\in(0,1)$, the maximum number of iterations $K$ per temperature, and a minimum temperature $T_{\text{min}}$. Specify a step size (learning rate) $\beta > 0$, set $\texttt{converged} \gets \texttt{false}$, set $x^{\star} \gets x_0$ as the best solution found so far, and $x_{c}\gets{x}_{0}$ as the current solution. Specify values for the penalty update parameters $(\tau_{\mu},\tau_{\rho})\in\left(0,1\right)$.

While not $\texttt{converged}$ __do__:

1. For $k = 1\,\text{to}\,K$:
   - Generate a _new_ candidate solution: $x^{\prime} \gets x_{c} + \beta\cdot\texttt{randn}(\texttt{size}(x_{c}))$.
   - Compute the change in the objective function between the __new__ solution and the __current__ solution: $\Delta P \gets P_{\mu,\rho}(x^{\prime}) - P_{\mu,\rho}(x_{c})$
      - _Downhill move_: If $\Delta P < 0$, accept the new solution (new solution becomes the current solution): $x_{c} \gets x^{\prime}$.
      - _Uphill move_: If $\Delta P \geq 0$, accept the new solution with probability $p\gets\exp(-\Delta P / T)$. Roll a uniform random number $u \gets \texttt{Uniform}(0,1)$. If $u \leq p$, accept the new solution: $x_c \gets x^{\prime}$. 
    - Compute the change in the objective function between the __current__ solution and the __best__ solution: $\Delta P^{\star} \gets  P_{\mu,\rho}(x_c) - P_{\mu,\rho}(x^{\star})$. If $\Delta P^{\star}<0$ update __best__ solution: $x^{\star}\gets{x}_{c}$.
2. Update the penalty parameters: $\mu\gets \tau_\mu\,\mu$ and $\rho\gets \tau_\rho\,\rho$.
3. Check for convergence: 
   - If $T \leq T_{\text{min}}$, set $\texttt{converged} \gets \texttt{true}$ and return the best solution $x^{\star}$.
   - Otherwise, update the temperature: $T \gets \alpha T$


### Selecting $T_{\circ}$ and $K$
A common question is how to choose the initial temperature $T_\circ$ and the number of iterations $K$ per temperature. Here are two standard heuristics:


#### Sample‐and‐set approach for $T_\circ$.
Let's look at a simple algorithm for selecting a $T_\circ$ value, called the _sample-and-set_ approach:
1. From the initial solution $x_\circ$ (and default values for the other parameters), generate $M={100}$ random neighbor costs $\Delta{P}_{1},\dots,\Delta{P}_{M}$.
2. Find $J^{+}\gets\left\{j: \Delta{P}_{j}>0,\,j=1,\dots,M\right\}$. Compute the mean _uphill_ neighbor costs for $\overline{\Delta P}_{+}\gets\texttt{mean}\left\{\Delta{P}_{i}\right\}_{i=J^{+}}$.
3. Choose an initial desired acceptance rate $p_{\circ}\in(0.6,0.9)$.
4. Set $T_{\circ}\gets{-{\overline{\Delta P}_{+}}/{\,\ln{p_{\circ}}}}$

#### Heuristic for choosing $K$
Next, let's sketch out a simple heuristic for choosing $K$ that updates the number of iterations per temperature as the simulated annealing algorithm proceeds.
1. Initially, choose $K\gets{c}\cdot\texttt{size}(x)$, where $c\in\left[10,50\right]$.
2. After each temperature, compute the fraction of _accepted_ moves $\hat{p}$.
    - If $\hat{p}>0.8$, increase $K\gets\lceil 1.5K\rceil$ (round to nearest integer value).
    - If $\hat{p}<{0.2}$, decrease $K\gets\lceil 0.75K\rceil$ (round to nearest integer value).


___